# Plagiarism Detection – Research Analysis

This notebook walks through a full research workflow:
1. Generate (or load) a PAN-style corpus
2. Run each detection algorithm
3. Evaluate with standard research metrics
4. Visualise results

In [ ]:
import sys, os
# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

## 1. Create a Sample PAN Corpus

In [ ]:
from src.pan_dataset_manager import PANDatasetManager

CORPUS_DIR = os.path.join(PROJECT_ROOT, 'data', 'pan_corpus', 'sample')

mgr = PANDatasetManager()
if not os.path.exists(CORPUS_DIR):
    mgr.create_sample_corpus(CORPUS_DIR)
    print(f'Sample corpus created at {CORPUS_DIR}')

pairs = mgr.load_corpus(CORPUS_DIR)
labels = [p['label'] for p in pairs]
print(f'Loaded {len(pairs)} pairs — {sum(labels)} plagiarised, {len(labels)-sum(labels)} clean')

## 2. Corpus Statistics

In [ ]:
stats = mgr.get_corpus_statistics(CORPUS_DIR)
for k, v in stats.items():
    print(f'  {k}: {v}')

## 3. Run Detection Algorithms

In [ ]:
from src.detection_algorithms import TFIDFAlgorithm, NGramAlgorithm, EmbeddingAlgorithm, EnsembleAlgorithm

text_pairs = [(p['text1'], p['text2']) for p in pairs]

algorithms = [
    TFIDFAlgorithm(),
    NGramAlgorithm(),
    EmbeddingAlgorithm(),
    EnsembleAlgorithm(),
]

all_results = {}
for algo in algorithms:
    results = algo.batch_detect(text_pairs)
    all_results[algo.name] = results
    preds = [int(r['is_plagiarized']) for r in results]
    print(f'{algo.name}: {sum(preds)}/{len(preds)} flagged as plagiarised')

## 4. Evaluate Algorithms

In [ ]:
from src.research_evaluator import ResearchEvaluator

evaluator = ResearchEvaluator()
comparison = {}
for name, results in all_results.items():
    preds = [int(r['is_plagiarized']) for r in results]
    scores = [r['similarity'] for r in results]
    metrics = evaluator.evaluate(preds, labels, scores=scores)
    comparison[name] = metrics

print(evaluator.format_results_table(comparison))

## 5. Similarity Score Distribution

In [ ]:
try:
    import matplotlib.pyplot as plt
    from src.utils.visualization import plot_similarity_distribution, plot_algorithm_comparison

    for name, results in all_results.items():
        similarities = [r['similarity'] for r in results]
        fig = plot_similarity_distribution(similarities, labels, title=f'{name} – Similarity Distribution')
        plt.show()

    fig = plot_algorithm_comparison(comparison, metric='f1')
    plt.show()
except ImportError:
    print('matplotlib not installed – skipping plots.')

## 6. Summary Statistics

In [ ]:
from src.utils.statistics import compute_summary_statistics

for name, results in all_results.items():
    sims = [r['similarity'] for r in results]
    s = compute_summary_statistics(sims)
    print(f'{name}: mean={s["mean"]:.3f}, std={s["std"]:.3f}, min={s["min"]:.3f}, max={s["max"]:.3f}')

## 7. Generate a Research Report

You can generate an HTML or text report from the notebook by calling the report generator directly:

In [ ]:
import json, subprocess

RESULTS_FILE = os.path.join(PROJECT_ROOT, 'data', 'results', 'notebook_benchmark.json')
REPORT_DIR = os.path.join(PROJECT_ROOT, 'data', 'reports')
os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# Serialise benchmark results
serialisable = {
    name: {
        'metrics': metrics,
        'predictions': [int(r['is_plagiarized']) for r in all_results[name]],
        'similarities': [r['similarity'] for r in all_results[name]],
    }
    for name, metrics in comparison.items()
}
with open(RESULTS_FILE, 'w') as f:
    json.dump(serialisable, f, indent=2)
print('Saved benchmark results to', RESULTS_FILE)

result = subprocess.run(
    [sys.executable, os.path.join(PROJECT_ROOT, 'scripts', 'generate_research_report.py'),
     '--results-file', RESULTS_FILE, '--output-dir', REPORT_DIR, '--format', 'html'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)